In [1]:
# ============================================================
# Phase 1 | Cell 1: Imports + Configuration
# ============================================================

import pandas as pd
import numpy as np
import requests
import warnings
import os
from fredapi import Fred

warnings.filterwarnings('ignore')

# ── Date range ───────────────────────────────────────────────
START = '2000-01-01'
END   = '2025-12-31'

# ── FRED API key ─────────────────────────────────────────────
FRED_API_KEY = __import__('os').environ.get('FRED_API_KEY', '')
fred = Fred(api_key=FRED_API_KEY)

# ── Paths ─────────────────────────────────────────────────────
RAW_PATH       = '../data/raw/'
PROCESSED_PATH = '../data/processed/'

os.makedirs(RAW_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)

print("✓ Environment ready")
print(f"  Date range : {START} → {END}")
print(f"  Raw path   : {RAW_PATH}")
print(f"  Processed  : {PROCESSED_PATH}")

✓ Environment ready
  Date range : 2000-01-01 → 2025-12-31
  Raw path   : ../data/raw/
  Processed  : ../data/processed/


In [2]:
# ============================================================
# Phase 1 | Cell 2: FRED Pulls (US + Global Series)
# ============================================================

def fetch_fred(series_id, name, freq='MS'):
    """Fetch a FRED series, resample to month-start, return named Series."""
    try:
        s = fred.get_series(series_id, observation_start=START, observation_end=END)
        s = s.resample(freq).last()
        s.index = s.index.to_period('M').to_timestamp()
        s.name = name
        print(f"  ✓ {name:<35} {len(s):>4} obs")
        return s
    except Exception as e:
        print(f"  ✗ {name:<35} FAILED — {e}")
        return None

print("Fetching FRED series...")
print("─" * 55)

fred_series = {
    # US / Global
    'CPIAUCSL'  : 'cpi_us',           # US CPI (All Urban Consumers)
    'FEDFUNDS'  : 'fed_funds_rate',   # Federal Funds Rate
    'DTWEXBGS'  : 'dxy',              # USD Broad Trade-Weighted Index
    'VIXCLS'    : 'vix',              # CBOE VIX
    'GS10'      : 'us_10y_yield',     # US 10Y Treasury Yield
    'DCOILBRENTEU' : 'brent',         # Brent Crude (USD/barrel)
    'WALCL'     : 'fed_balance_sheet',# Fed Balance Sheet (Total Assets)
}

frames = []
for series_id, name in fred_series.items():
    s = fetch_fred(series_id, name)
    if s is not None:
        frames.append(s)

df_fred = pd.concat(frames, axis=1)
df_fred.index.name = 'date'

# Save raw
df_fred.to_csv(f'{RAW_PATH}fred_raw.csv')

print("─" * 55)
print(f"\n✓ FRED pull complete")
print(f"  Shape  : {df_fred.shape}")
print(f"  Period : {df_fred.index[0].strftime('%b %Y')} → {df_fred.index[-1].strftime('%b %Y')}")
print(f"\nMissing values per series:")
print(df_fred.isnull().sum().to_string())

Fetching FRED series...
───────────────────────────────────────────────────────
  ✓ cpi_us                               312 obs
  ✓ fed_funds_rate                       312 obs
  ✓ dxy                                  240 obs
  ✓ vix                                  312 obs
  ✓ us_10y_yield                         312 obs
  ✓ brent                                312 obs
  ✓ fed_balance_sheet                    277 obs
───────────────────────────────────────────────────────

✓ FRED pull complete
  Shape  : (312, 7)
  Period : Jan 2000 → Dec 2025

Missing values per series:
cpi_us                1
fed_funds_rate        0
dxy                  72
vix                   0
us_10y_yield          0
brent                 0
fed_balance_sheet    35


In [3]:
# ============================================================
# Phase 1 | Cell 3: Domestic Series (RBI + World Bank)
# ============================================================

import requests
from io import StringIO

# ── Part A: World Bank API (INR/USD, CPI India, GDP) ─────────
print("Fetching World Bank series...")
print("─" * 55)

WB_BASE = "https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"
WB_PARAMS = {
    'format'   : 'json',
    'per_page' : '1000',
    'mrv'      : '30',
    'date'     : '2000:2025',
}

def fetch_worldbank(country, indicator, name):
    url = WB_BASE.format(country=country, indicator=indicator)
    try:
        r = requests.get(url, params=WB_PARAMS, timeout=15)
        data = r.json()[1]
        records = {
            pd.Timestamp(d['date'] + '-01-01'): d['value']
            for d in data if d['value'] is not None
        }
        s = pd.Series(records, name=name).sort_index()
        # Resample to annual, will be forward-filled to monthly later
        print(f"  ✓ {name:<35} {len(s):>4} obs (annual)")
        return s
    except Exception as e:
        print(f"  ✗ {name:<35} FAILED — {e}")
        return None

wb_gdp_india  = fetch_worldbank('IN', 'NY.GDP.MKTP.CD',  'gdp_india_usd')
wb_gdp_us     = fetch_worldbank('US', 'NY.GDP.MKTP.CD',  'gdp_us_usd')
wb_gdp_pc_in  = fetch_worldbank('IN', 'NY.GDP.PCAP.CD',  'gdp_pc_india')
wb_gdp_pc_us  = fetch_worldbank('US', 'NY.GDP.PCAP.CD',  'gdp_pc_us')

print()

# ── Part B: FRED India-related series ────────────────────────
print("Fetching FRED India-related series...")
print("─" * 55)

india_fred = {
    'INDCPIALLMINMEI' : 'cpi_india',        # India CPI (OECD via FRED)
    'INTDSRINM193N'   : 'rbi_repo_rate',    # RBI Policy Rate
    'IRSTCI01INM156N' : 'india_stir',       # India Short-term Interest Rate
}

india_frames = []
for series_id, name in india_fred.items():
    s = fetch_fred(series_id, name)
    if s is not None:
        india_frames.append(s)

df_india_fred = pd.concat(india_frames, axis=1) if india_frames else pd.DataFrame()
print()

# ── Part C: Manual Download Instructions ─────────────────────
print("─" * 55)
print("MANUAL DOWNLOADS REQUIRED (RBI DBIE)")
print("─" * 55)
print("""
The following series must be downloaded manually from RBI DBIE.
URL: https://dbie.rbi.org.in/DBIE/dbie.rbi?site=publications

Download each as CSV and save to: ../data/raw/

  1. INR/USD Exchange Rate (monthly average)
     → Database > Financial Markets > Exchange Rates
     → File: rbi_inr_usd.csv

  2. Foreign Exchange Reserves (USD million)  
     → Database > External Sector > Foreign Exchange Reserves
     → File: rbi_reserves.csv

  3. Current Account Balance (USD million, quarterly)
     → Database > External Sector > Balance of Payments
     → File: rbi_cab.csv

  4. FPI/FII Flows - Equity + Debt (USD million, monthly)
     → Database > Financial Markets > FII/FPI Flows  
     → File: rbi_fpi_flows.csv

  5. Merchandise Exports + Imports (USD million, monthly)
     → Database > External Sector > Foreign Trade
     → File: rbi_trade.csv

  6. REER - 36 currency index (monthly)
     → Database > External Sector > REER/NEER
     → File: rbi_reer.csv

  7. Inward Remittances (USD million, annual/quarterly)
     → Database > External Sector > Invisibles
     → File: rbi_remittances.csv
""")

# ── Save what we have ─────────────────────────────────────────
df_india_fred.to_csv(f'{RAW_PATH}india_fred_raw.csv')

for name, s in [('wb_gdp_india', wb_gdp_india), ('wb_gdp_us', wb_gdp_us),
                ('wb_gdp_pc_india', wb_gdp_pc_in), ('wb_gdp_pc_us', wb_gdp_pc_us)]:
    if s is not None:
        s.to_csv(f'{RAW_PATH}{name}.csv')

print("✓ Programmatic pulls saved to raw/")
print("✓ Begin manual RBI downloads while reviewing output above")

Fetching World Bank series...
───────────────────────────────────────────────────────
  ✓ gdp_india_usd                         30 obs (annual)
  ✓ gdp_us_usd                            30 obs (annual)
  ✓ gdp_pc_india                          30 obs (annual)
  ✓ gdp_pc_us                             30 obs (annual)

Fetching FRED India-related series...
───────────────────────────────────────────────────────
  ✓ cpi_india                            303 obs
  ✓ rbi_repo_rate                        271 obs
  ✓ india_stir                           312 obs

───────────────────────────────────────────────────────
MANUAL DOWNLOADS REQUIRED (RBI DBIE)
───────────────────────────────────────────────────────

The following series must be downloaded manually from RBI DBIE.
URL: https://dbie.rbi.org.in/DBIE/dbie.rbi?site=publications

Download each as CSV and save to: ../data/raw/

  1. INR/USD Exchange Rate (monthly average)
     → Database > Financial Markets > Exchange Rates
     → File: rbi_

In [7]:
wb_remittances = fetch_worldbank('IN', 'BX.TRF.PWKR.CD.DT', 'remittances_inward')
wb_remittances.to_csv(f'{RAW_PATH}wb_remittances.csv')
print(f"✓ remittances_inward saved → {RAW_PATH}wb_remittances.csv")

  ✓ remittances_inward                    30 obs (annual)
✓ remittances_inward saved → ../data/raw/wb_remittances.csv


In [12]:
import os
files = os.listdir('../data/raw/')
for f in sorted(files):
    size = os.path.getsize(f'../data/raw/{f}')
    print(f"  {f:<35} {size:>8,} bytes")

  fred_raw.csv                          17,723 bytes
  india_fred_raw.csv                     9,472 bytes
  rbi_cab.xlsx                         172,016 bytes
  rbi_fpi_flows.xlsx                     9,674 bytes
  rbi_inr_usd.xlsx                      43,711 bytes
  rbi_reer.xlsx                         18,145 bytes
  rbi_reserves.xlsx                     43,620 bytes
  rbi_trade.xlsx                        23,172 bytes
  wb_gdp_india.csv                         884 bytes
  wb_gdp_pc_india.csv                      884 bytes
  wb_gdp_pc_us.csv                         878 bytes
  wb_gdp_us.csv                            878 bytes
  wb_remittances.csv                       889 bytes


In [2]:
# ============================================================
# Phase 1 | Cell 3c: Excel Structure Inspection
# ============================================================

import openpyxl

# UPDATED: Changed .csv to .xlsx to match your actual files
rbi_files = [
    'rbi_inr_usd.xlsx',
    'rbi_reer.xlsx',
    'rbi_reserves.xlsx',
    'rbi_cab.xlsx',
    'rbi_fpi_flows.xlsx',
    'rbi_trade.xlsx',
]

for fname in rbi_files:
    path = f'../data/raw/{fname}'
    print(f"\n{'='*60}")
    print(f"FILE: {fname}")
    print('='*60)
    try:
        wb = openpyxl.load_workbook(path, data_only=True)
        print(f"  Sheets: {wb.sheetnames}")
        ws = wb.active
        print(f"  Dimensions: {ws.dimensions}")
        print(f"  First 10 rows:")
        for i, row in enumerate(ws.iter_rows(min_row=1, max_row=10, values_only=True)):
            print(f"    row {i:02d}: {row}")
    except Exception as e:
        print(f"  ERROR: {e}")


FILE: rbi_inr_usd.xlsx
  Sheets: ['EXCHANGE RATE OF THE INDIAN RUP']
  Dimensions: A1:M419
  First 10 rows:
    row 00: ('Exchange Rate of the Indian Rupee vis-à-vis the SDR, US Dollar, Pound Sterling, Euro and Japanese Yen (Monthly Average and end-Month Rates)', None, None, None, None, None, None, None, None, None, None, None, None)
    row 01: (None, None, None, None, None, None, None, None, None, None, None, None, None)
    row 02: ('(₹ per unit of foreign currency)', None, None, None, None, None, None, None, None, None, None, None, None)
    row 03: (None, None, None, None, None, None, None, None, None, None, None, None, None)
    row 04: (None, 'Year/Month', 'SDR', None, 'US Dollar', None, 'Pound Sterling', None, 'Japanese Yen', None, 'Euro', None, None)
    row 05: (None, None, 'Average', 'End-month', 'Average', 'End-month', 'Average', 'End-month', 'Average', 'End-month', 'Average', 'End-month', None)
    row 06: (None, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, None)
    row 07: (datet

In [3]:
import dill
import os

# Ensure outputs directory exists just in case
os.makedirs('../outputs', exist_ok=True)

# Define the specific save path for this notebook
session_file = '../outputs/01_data_build_session.pkl'

# Save the environment
dill.dump_session(session_file)
print(f"✅ Workspace for '01_data_build' successfully saved to: {session_file}")

✅ Workspace for '01_data_build' successfully saved to: ../outputs/01_data_build_session.pkl


In [2]:
import dill
import os

# 1. Clean the environment of unpicklable objects!
try:
    del wb, ws, openpyxl
except NameError:
    pass # They weren't in memory anyway, which is fine

# 2. Save the session safely
os.makedirs('../outputs', exist_ok=True)
session_file = '../outputs/01_data_build_session.pkl'

dill.dump_session(session_file)
print(f"✅ Workspace successfully saved to: {session_file}")

✅ Workspace successfully saved to: ../outputs/01_data_build_session.pkl


In [4]:
# ============================================================
# Phase 1 | Cell 4: Parse + Clean All RBI Files
# ============================================================

import openpyxl
import pandas as pd
import numpy as np
from datetime import datetime

RAW  = '../data/raw/'
month_map = {
    'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,
    'JUL':7,'AUG':8,'SEP':9,'OCT':10,'NOV':11,'DEC':12,
    'JANUARY':1,'FEBRUARY':2,'MARCH':3,'APRIL':4,'JUNE':6,
    'JULY':7,'AUGUST':8,'SEPTEMBER':9,'OCTOBER':10,
    'NOVEMBER':11,'DECEMBER':12
}

# ── Helper ────────────────────────────────────────────────────
def to_monthly_index(s, freq='MS'):
    s.index = pd.to_datetime(s.index)
    s = s.sort_index()
    s = s[s.index >= '2000-01-01']
    s = s.resample(freq).last()
    s.index = s.index.to_period('M').to_timestamp()
    return s

print("Parsing RBI files...")
print("─" * 55)

# ── 1. INR/USD ────────────────────────────────────────────────
wb  = openpyxl.load_workbook(f'{RAW}rbi_inr_usd.xlsx', data_only=True)
ws  = wb.active
rows = []
for row in ws.iter_rows(min_row=8, values_only=True):
    date, usd_avg = row[0], row[4]
    if isinstance(date, datetime) and isinstance(usd_avg, (int, float)):
        rows.append((date, float(usd_avg)))

inr_usd = pd.Series(
    {r[0]: r[1] for r in rows}, name='inr_usd'
)
inr_usd = to_monthly_index(inr_usd)
print(f"  ✓ inr_usd          {len(inr_usd):>4} obs  "
      f"[{inr_usd.index[0].strftime('%b %Y')} → {inr_usd.index[-1].strftime('%b %Y')}]")

# ── 2. REER ───────────────────────────────────────────────────
wb  = openpyxl.load_workbook(f'{RAW}rbi_reer.xlsx', data_only=True)
ws  = wb.active
rows = []
for row in ws.iter_rows(min_row=7, values_only=True):
    date, neer, reer = row[0], row[1], row[2]
    if isinstance(date, datetime) and isinstance(reer, (int, float)):
        rows.append((date, float(neer), float(reer)))

reer_df = pd.DataFrame(rows, columns=['date','neer','reer']).set_index('date')
reer_df = reer_df[reer_df.index >= '2000-01-01'].sort_index()
reer_df.index = pd.to_datetime(reer_df.index)
reer_df = reer_df.resample('MS').last()
reer_df.index = reer_df.index.to_period('M').to_timestamp()
print(f"  ✓ reer/neer        {len(reer_df):>4} obs  "
      f"[{reer_df.index[0].strftime('%b %Y')} → {reer_df.index[-1].strftime('%b %Y')}]")

# ── 3. Reserves ───────────────────────────────────────────────
wb  = openpyxl.load_workbook(f'{RAW}rbi_reserves.xlsx', data_only=True)
ws  = wb.active
rows = []
current_year = None
for row in ws.iter_rows(min_row=6, values_only=True):
    year_val, month_val, total_usd = row[0], row[1], row[12]
    if isinstance(year_val, str) and year_val.strip().isdigit():
        current_year = int(year_val.strip())
    if current_year and isinstance(month_val, str):
        m = month_map.get(month_val.strip().upper())
        if m and isinstance(total_usd, (int, float)):
            rows.append((pd.Timestamp(year=current_year, month=m, day=1), float(total_usd)))

reserves = pd.Series(
    {r[0]: r[1] for r in rows}, name='fx_reserves_usd_mn'
)
reserves = to_monthly_index(reserves)
print(f"  ✓ fx_reserves      {len(reserves):>4} obs  "
      f"[{reserves.index[0].strftime('%b %Y')} → {reserves.index[-1].strftime('%b %Y')}]")

# ── 4. Current Account Balance (Quarterly → Monthly) ─────────
wb  = openpyxl.load_workbook(f'{RAW}rbi_cab.xlsx', data_only=True)
ws  = wb.active
rows = []
current_fy = None
quarter_month_map = {'Q1': 4, 'Q2': 7, 'Q3': 10, 'Q4': 1}

for row in ws.iter_rows(min_row=6, values_only=True):
    fy, qtr, txn_type, cab = row[0], row[1], row[2], row[4]
    if isinstance(fy, str) and '-' in fy:
        current_fy = fy.strip()
    if (current_fy and isinstance(qtr, str) and
            qtr.strip() in quarter_month_map and
            isinstance(txn_type, str) and
            txn_type.strip().lower() == 'net' and
            isinstance(cab, (int, float))):
        q     = qtr.strip()
        month = quarter_month_map[q]
        fy_start = int(current_fy.split('-')[0])
        year  = fy_start if month >= 4 else fy_start + 1
        rows.append((pd.Timestamp(year=year, month=month, day=1), float(cab)))

cab = pd.Series(
    {r[0]: r[1] for r in rows}, name='current_account_usd_mn'
).sort_index()
cab = cab[cab.index >= '2000-01-01']
# Forward-fill quarterly to monthly
cab = cab.resample('MS').last().ffill()
cab.index = cab.index.to_period('M').to_timestamp()
print(f"  ✓ current_account  {len(cab):>4} obs  "
      f"[{cab.index[0].strftime('%b %Y')} → {cab.index[-1].strftime('%b %Y')}]")

# ── 5. FPI Flows (Wide FY format → Long monthly, convert to USD mn) ──
wb  = openpyxl.load_workbook(f'{RAW}rbi_fpi_flows.xlsx', data_only=True)
ws  = wb.active

col_months = ['Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar']
rows = []

for row in ws.iter_rows(min_row=7, values_only=True):
    fy_label = row[0]
    if not isinstance(fy_label, str) or '-' not in fy_label:
        continue
    fy_label = fy_label.strip()
    try:
        fy_start = int(fy_label.split('-')[0])
    except:
        continue
    monthly_vals = list(row[1:13])
    for i, val in enumerate(monthly_vals):
        if val is None:
            continue
        m_name = col_months[i]
        month  = month_map[m_name.upper()]
        year   = fy_start if month >= 4 else fy_start + 1
        rows.append((pd.Timestamp(year=year, month=month, day=1), float(val)))

fpi_inr = pd.Series(
    {r[0]: r[1] for r in rows}, name='fpi_flows_inr_cr'
).sort_index()
fpi_inr = fpi_inr[fpi_inr.index >= '2000-01-01']
fpi_inr = fpi_inr.resample('MS').last()
fpi_inr.index = fpi_inr.index.to_period('M').to_timestamp()

# Convert INR Crores → USD Millions using monthly INR/USD
# INR Crores × 10 = INR Millions ÷ INR/USD rate = USD Millions
fpi_usd = (fpi_inr * 10) / inr_usd
fpi_usd.name = 'fpi_flows_usd_mn'

# Keep both — INR for reference, USD for modeling
fpi_inr.to_csv(f'{RAW}parsed_fpi_inr.csv')
fpi_usd.to_csv(f'{RAW}parsed_fpi_usd.csv')

print(f"  ✓ fpi_flows (INR)  {len(fpi_inr):>4} obs  "
      f"[{fpi_inr.index[0].strftime('%b %Y')} → {fpi_inr.index[-1].strftime('%b %Y')}]")
print(f"  ✓ fpi_flows (USD)  {len(fpi_usd.dropna()):>4} obs  "
      f"(after INR/USD conversion)")


# ── 6. Trade (Exports + Imports) ─────────────────────────────
wb  = openpyxl.load_workbook(f'{RAW}rbi_trade.xlsx', data_only=True)
ws  = wb.active
rows = []
current_fy = None

for row in ws.iter_rows(min_row=6, values_only=True):
    fy_val, month_val, exports, imports = row[0], row[1], row[2], row[3]
    if isinstance(fy_val, str) and '-' in fy_val:
        current_fy = fy_val.strip()
    if (current_fy and isinstance(month_val, str) and
            isinstance(exports, (int, float)) and
            isinstance(imports, (int, float))):
        m = month_map.get(month_val.strip().upper())
        if m:
            fy_start = int(current_fy.split('-')[0])
            year = fy_start if m >= 4 else fy_start + 1
            rows.append((
                pd.Timestamp(year=year, month=m, day=1),
                float(exports), float(imports)
            ))

trade_df = pd.DataFrame(
    rows, columns=['date','exports_usd_mn','imports_usd_mn']
).set_index('date').sort_index()
trade_df = trade_df[trade_df.index >= '2000-01-01']
trade_df = trade_df.resample('MS').last()
trade_df.index = trade_df.index.to_period('M').to_timestamp()
print(f"  ✓ trade            {len(trade_df):>4} obs  "
      f"[{trade_df.index[0].strftime('%b %Y')} → {trade_df.index[-1].strftime('%b %Y')}]")

# ── Save parsed files ─────────────────────────────────────────
inr_usd.to_csv(f'{RAW}parsed_inr_usd.csv')
reer_df.to_csv(f'{RAW}parsed_reer.csv')
reserves.to_csv(f'{RAW}parsed_reserves.csv')
cab.to_csv(f'{RAW}parsed_cab.csv')
fpi_inr.to_csv(f'{RAW}parsed_fpi_inr.csv')
fpi_usd.to_csv(f'{RAW}parsed_fpi_usd.csv')
trade_df.to_csv(f'{RAW}parsed_trade.csv')

print("─" * 55)
print("✓ All files parsed and saved to raw/")

Parsing RBI files...
───────────────────────────────────────────────────────
  ✓ inr_usd           316 obs  [Jan 2000 → Apr 2026]
  ✓ reer/neer         265 obs  [Apr 2004 → Apr 2026]
  ✓ fx_reserves       314 obs  [Jan 2000 → Feb 2026]
  ✓ current_account   307 obs  [Apr 2000 → Oct 2025]
  ✓ fpi_flows (INR)   306 obs  [Jan 2000 → Jun 2025]
  ✓ fpi_flows (USD)   306 obs  (after INR/USD conversion)
  ✓ trade             315 obs  [Jan 2000 → Mar 2026]
───────────────────────────────────────────────────────
✓ All files parsed and saved to raw/


In [13]:
# ============================================================
# Phase 1 | Cell 5: Master DataFrame Assembly + Derived Series
# ============================================================

import pandas as pd
import numpy as np

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'

print("Assembling master DataFrame...")
print("─" * 55)

# ── Load all programmatic pulls ───────────
# FRED needs dayfirst=True to stop MM/DD vs DD/MM confusion
df_fred       = pd.read_csv(f'{RAW}fred_raw.csv',        index_col=0, parse_dates=True, dayfirst=True)
df_india_fred = pd.read_csv(f'{RAW}india_fred_raw.csv',  index_col=0, parse_dates=True, dayfirst=True)

# World Bank and RBI are natively formatted correctly - DO NOT use dayfirst=True here
wb_gdp_india  = pd.read_csv(f'{RAW}wb_gdp_india.csv',    index_col=0, parse_dates=True).squeeze()
wb_gdp_us     = pd.read_csv(f'{RAW}wb_gdp_us.csv',       index_col=0, parse_dates=True).squeeze()
wb_gdp_pc_in  = pd.read_csv(f'{RAW}wb_gdp_pc_india.csv', index_col=0, parse_dates=True).squeeze()
wb_gdp_pc_us  = pd.read_csv(f'{RAW}wb_gdp_pc_us.csv',    index_col=0, parse_dates=True).squeeze()
wb_remit      = pd.read_csv(f'{RAW}wb_remittances.csv',  index_col=0, parse_dates=True).squeeze()

# RBI parsed series
inr_usd   = pd.read_csv(f'{RAW}parsed_inr_usd.csv',   index_col=0, parse_dates=True).squeeze()
reer_df   = pd.read_csv(f'{RAW}parsed_reer.csv',       index_col=0, parse_dates=True)
reserves  = pd.read_csv(f'{RAW}parsed_reserves.csv',   index_col=0, parse_dates=True).squeeze()
cab       = pd.read_csv(f'{RAW}parsed_cab.csv',        index_col=0, parse_dates=True).squeeze()
fpi_usd   = pd.read_csv(f'{RAW}parsed_fpi_usd.csv',   index_col=0, parse_dates=True, dayfirst=True).squeeze()
fpi_inr   = pd.read_csv(f'{RAW}parsed_fpi_inr.csv',   index_col=0, parse_dates=True).squeeze()
trade_df  = pd.read_csv(f'{RAW}parsed_trade.csv',      index_col=0, parse_dates=True)

# ── Standardise indices to month-start & Deduplicate ──────────
def force_monthly(data):
    """Safely converts index to tz-naive month-start and removes duplicates"""
    data.index = pd.to_datetime(data.index, utc=True).tz_localize(None)
    data.index = data.index.to_period('M').to_timestamp()
    return data[~data.index.duplicated(keep='last')]

# Clean every source before assembly
df_fred       = force_monthly(df_fred)
df_india_fred = force_monthly(df_india_fred)
inr_usd       = force_monthly(inr_usd)
reer_df       = force_monthly(reer_df)
reserves      = force_monthly(reserves)
cab           = force_monthly(cab)
fpi_usd       = force_monthly(fpi_usd)
fpi_inr       = force_monthly(fpi_inr)
trade_df      = force_monthly(trade_df)

# ── Upsample annual World Bank series to monthly ──────────────
master_idx = pd.date_range('2000-01-01', '2025-12-01', freq='MS')

def annual_to_monthly(s, idx):
    s.index = pd.to_datetime(s.index).to_period('M').to_timestamp()
    s = s[~s.index.duplicated(keep='last')]
    s = s.reindex(idx).ffill()
    return s

wb_gdp_india_m = annual_to_monthly(wb_gdp_india,  master_idx)
wb_gdp_us_m    = annual_to_monthly(wb_gdp_us,     master_idx)
wb_gdp_pc_in_m = annual_to_monthly(wb_gdp_pc_in,  master_idx)
wb_gdp_pc_us_m = annual_to_monthly(wb_gdp_pc_us,  master_idx)
wb_remit_m     = annual_to_monthly(wb_remit,      master_idx)

# ── DXY backfill: DTWEXBGS starts 2006, DTWEXM covers 2000─────
# Pull narrow DXY (DTWEXM) for 2000-2005 gap
try:
    from fredapi import Fred
    import os
    fred = Fred(api_key=os.environ.get('FRED_API_KEY', FRED_API_KEY))
    dxy_narrow = fred.get_series('DTWEXM',
                                  observation_start='2000-01-01',
                                  observation_end='2006-01-01')
    dxy_narrow = dxy_narrow.resample('MS').last()
    dxy_narrow.index = dxy_narrow.index.to_period('M').to_timestamp()
    dxy_broad = df_fred['dxy'].copy()
    dxy_combined = dxy_broad.combine_first(dxy_narrow)
    dxy_combined.name = 'dxy'
    print("  ✓ DXY backfilled to Jan 2000 using DTWEXM")
except Exception as e:
    dxy_combined = df_fred['dxy'].copy()
    print(f"  ⚠ DXY backfill failed ({e}) — using available data")

# ── Assemble master DataFrame ─────────────────────────────────
master = pd.DataFrame(index=master_idx)

# Market / spot
master['inr_usd']            = inr_usd
master['reer']               = reer_df['reer']
master['neer']               = reer_df['neer']

# US / Global
master['cpi_us']             = df_fred['cpi_us']
master['fed_funds_rate']     = df_fred['fed_funds_rate']
master['dxy']                = dxy_combined
master['vix']                = df_fred['vix']
master['us_10y_yield']       = df_fred['us_10y_yield']
master['brent']              = df_fred['brent']
master['fed_balance_sheet']  = df_fred['fed_balance_sheet']

# India macro
master['cpi_india']          = df_india_fred['cpi_india']
master['rbi_repo_rate']      = df_india_fred['rbi_repo_rate']
master['india_stir']         = df_india_fred['india_stir']
master['fx_reserves_usd_mn'] = reserves
master['current_account']    = cab
master['fpi_flows_usd_mn']   = fpi_usd
master['fpi_flows_inr_cr']   = fpi_inr
master['exports_usd_mn']     = trade_df['exports_usd_mn']
master['imports_usd_mn']     = trade_df['imports_usd_mn']

# World Bank
master['gdp_india_usd']      = wb_gdp_india_m
master['gdp_us_usd']         = wb_gdp_us_m
master['gdp_pc_india']       = wb_gdp_pc_in_m
master['gdp_pc_us']          = wb_gdp_pc_us_m
master['remittances_usd']    = wb_remit_m

print(f"  ✓ Master DataFrame assembled: {master.shape}")

# ── Derived Series ────────────────────────────────────────────
print("\nComputing derived series...")

# 1. Inflation differential (India minus US, YoY)
master['cpi_india_yoy']   = master['cpi_india'].pct_change(12, fill_method=None) * 100
master['cpi_us_yoy']      = master['cpi_us'].pct_change(12, fill_method=None) * 100
master['inflation_diff']  = master['cpi_india_yoy'] - master['cpi_us_yoy']

# 2. Real interest rate differential
master['real_rate_india'] = master['rbi_repo_rate'] - master['cpi_india_yoy']
master['real_rate_us']    = master['fed_funds_rate'] - master['cpi_us_yoy']
master['real_rate_diff']  = master['real_rate_india'] - master['real_rate_us']

# 3. Productivity differential (GDP per capita growth, annual proxy)
master['prod_india_yoy']  = master['gdp_pc_india'].pct_change(12, fill_method=None) * 100
master['prod_us_yoy']     = master['gdp_pc_us'].pct_change(12, fill_method=None) * 100
master['prod_diff']       = master['prod_india_yoy'] - master['prod_us_yoy']

# 4. Justified depreciation trend (Balassa-Samuelson adjustment)
master['justified_depreciation'] = master['inflation_diff'] - master['prod_diff']

# 5. Trade balance
master['trade_balance']   = master['exports_usd_mn'] - master['imports_usd_mn']

# 6. Import cover ratio (months of imports covered by reserves)
master['import_cover']    = master['fx_reserves_usd_mn'] / master['imports_usd_mn']

# 7. Oil import bill proxy (India imports ~85% of oil needs)
master['brent_yoy']       = master['brent'].pct_change(12, fill_method=None) * 100

# 8. CAD as % of GDP (quarterly CAD annualised / annual GDP)
master['cab_pct_gdp']     = ((master['current_account'] * 4) / (master['gdp_india_usd'] / 1e6)) * 100

# 9. Remittances as % of GDP
master['remit_pct_gdp']   = (master['remittances_usd'] / master['gdp_india_usd']) * 100

# 10. Fed balance sheet YoY growth
master['fed_bs_yoy']      = master['fed_balance_sheet'].pct_change(12, fill_method=None) * 100

# 11. INR YoY depreciation (positive = weaker INR)
master['inr_yoy_chg']     = master['inr_usd'].pct_change(12, fill_method=None) * 100

# 12. Implied RBI intervention proxy
master['reserves_mom_chg'] = master['fx_reserves_usd_mn'].diff()

# ── Trim to clean window ──────────────────────────────────────
master = master['2000-01-01':'2025-12-01']

# ── Validation Report ─────────────────────────────────────────
print("\n" + "─" * 55)
print("VALIDATION REPORT")
print("─" * 55)
print(f"  Shape          : {master.shape}")
print(f"  Date range     : {master.index[0].strftime('%b %Y')} → "
      f"{master.index[-1].strftime('%b %Y')}")
print(f"\n  Missing values per column:")
nulls = master.isnull().sum()
for col, n in nulls.items():
    pct   = n / len(master) * 100
    flag  = '  ⚠' if pct > 20 else '  ✓'
    print(f"  {flag} {col:<30} {n:>4} nulls ({pct:.1f}%)")

# ── Save master ───────────────────────────────────────────────
master.to_csv(f'{PROCESSED}master.csv')
print(f"\n✓ Master DataFrame saved → {PROCESSED}master.csv")
print(f"  {master.shape[0]} months × {master.shape[1]} columns")

Assembling master DataFrame...
───────────────────────────────────────────────────────
  ✓ DXY backfilled to Jan 2000 using DTWEXM
  ✓ Master DataFrame assembled: (312, 24)

Computing derived series...

───────────────────────────────────────────────────────
VALIDATION REPORT
───────────────────────────────────────────────────────
  Shape          : (312, 42)
  Date range     : Jan 2000 → Dec 2025

  Missing values per column:
    ✓ inr_usd                           0 nulls (0.0%)
    ✓ reer                             51 nulls (16.3%)
    ✓ neer                             51 nulls (16.3%)
    ✓ cpi_us                            1 nulls (0.3%)
    ✓ fed_funds_rate                    0 nulls (0.0%)
    ✓ dxy                               0 nulls (0.0%)
    ✓ vix                               0 nulls (0.0%)
    ✓ us_10y_yield                      0 nulls (0.0%)
    ✓ brent                             0 nulls (0.0%)
    ✓ fed_balance_sheet                35 nulls (11.2%)
    ✓ cpi_india 

In [2]:
# ============================================================
# Phase 1 | Cell 5b: Patches + Final Save
# ============================================================

master = pd.read_csv('../data/processed/master.csv',
                     index_col=0, parse_dates=True)
master.index = pd.to_datetime(master.index)

# ── Patch 1: repo rate gap ────────────────────────────────────
gap_mask = master['rbi_repo_rate'].isnull()
master.loc[gap_mask, 'rbi_repo_rate'] = master.loc[gap_mask, 'india_stir']
master['real_rate_india'] = master['rbi_repo_rate'] - master['cpi_india_yoy']
master['real_rate_diff']  = master['real_rate_india'] - master['real_rate_us']
print(f"✓ Repo rate: {gap_mask.sum()} months patched from india_stir")
print(f"  rbi_repo_rate now ends: {master['rbi_repo_rate'].dropna().index[-1].strftime('%b %Y')}")
print(f"  real_rate_diff nulls:   {master['real_rate_diff'].isnull().sum()}")

# ── Patch 2: re-check CPI india ──────────────────────────────
from fredapi import Fred
fred = Fred(api_key=FRED_API_KEY)
cpi_refresh = fred.get_series('INDCPIALLMINMEI',
                               observation_start='2000-01-01',
                               observation_end='2025-12-31')
cpi_refresh = cpi_refresh.resample('MS').last()
cpi_refresh.index = cpi_refresh.index.to_period('M').to_timestamp()
cpi_refresh.name = 'cpi_india'

# Only fill missing months — don't overwrite existing
gap_cpi = master['cpi_india'].isnull()
master.loc[gap_cpi, 'cpi_india'] = cpi_refresh[gap_cpi]
master['cpi_india_yoy'] = master['cpi_india'].pct_change(12, fill_method=None) * 100
master['inflation_diff'] = master['cpi_india_yoy'] - master['cpi_us_yoy']
master['justified_depreciation'] = master['inflation_diff'] - master['prod_diff']
master['real_rate_india'] = master['rbi_repo_rate'] - master['cpi_india_yoy']
master['real_rate_diff']  = master['real_rate_india'] - master['real_rate_us']
print(f"\n✓ CPI India refresh attempted")
print(f"  cpi_india now ends: {master['cpi_india'].dropna().index[-1].strftime('%b %Y')}")
print(f"  cpi_india nulls:    {master['cpi_india'].isnull().sum()}")

# ── Final null summary ────────────────────────────────────────
print(f"\n{'─'*55}")
print("FINAL NULL SUMMARY (changed columns only)")
print(f"{'─'*55}")
for col in ['rbi_repo_rate','real_rate_india','real_rate_diff',
            'cpi_india','cpi_india_yoy','inflation_diff','justified_depreciation']:
    n = master[col].isnull().sum()
    end = master[col].dropna().index[-1].strftime('%b %Y')
    print(f"  {col:<30} {n:>3} nulls  ends {end}")

# ── Overwrite master ──────────────────────────────────────────
master.to_csv('../data/processed/master.csv')
print(f"\n✓ Master overwritten → ../data/processed/master.csv")

✓ Repo rate: 41 months patched from india_stir
  rbi_repo_rate now ends: Dec 2025
  real_rate_diff nulls:   21

✓ CPI India refresh attempted
  cpi_india now ends: Mar 2025
  cpi_india nulls:    9

───────────────────────────────────────────────────────
FINAL NULL SUMMARY (changed columns only)
───────────────────────────────────────────────────────
  rbi_repo_rate                    0 nulls  ends Dec 2025
  real_rate_india                 21 nulls  ends Mar 2025
  real_rate_diff                  21 nulls  ends Mar 2025
  cpi_india                        9 nulls  ends Mar 2025
  cpi_india_yoy                   21 nulls  ends Mar 2025
  inflation_diff                  21 nulls  ends Mar 2025
  justified_depreciation          21 nulls  ends Mar 2025

✓ Master overwritten → ../data/processed/master.csv


In [3]:
# ============================================================
# Phase 1 | Cell 6: COMPLETE CLEAN REBUILD
# Full FRED refresh + CPI splice — no dependency on prior cells
# ============================================================

import pandas as pd
import numpy as np
import openpyxl
from fredapi import Fred
from datetime import datetime

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'

FRED_API_KEY = __import__('os').environ.get('FRED_API_KEY', '')
fred  = Fred(api_key=FRED_API_KEY)
START = '2000-01-01'
END   = '2026-05-01'

# ── Helpers ───────────────────────────────────────────────────
def to_ms(s):
    s = s.resample('MS').last()
    s.index = s.index.to_period('M').to_timestamp()
    return s

def force_ms(df):
    df.index = pd.to_datetime(df.index)
    df.index = df.index.to_period('M').to_timestamp()
    return df[~df.index.duplicated(keep='last')]

def get_val(df, dt, col):
    """Safe scalar extraction — handles duplicate index edge cases."""
    if dt not in df.index:
        return np.nan
    v = df.loc[dt, col]
    if isinstance(v, pd.Series):
        v = v.iloc[-1]
    return float(v) if pd.notna(v) else np.nan

month_map = {
    'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,
    'JUL':7,'AUG':8,'SEP':9,'OCT':10,'NOV':11,'DEC':12,
    'JANUARY':1,'FEBRUARY':2,'MARCH':3,'APRIL':4,'JUNE':6,
    'JULY':7,'AUGUST':8,'SEPTEMBER':9,'OCTOBER':10,
    'NOVEMBER':11,'DECEMBER':12
}

# ── Master index: monthly Jan 2000 → Apr 2026 ─────────────────
master_idx = pd.date_range(START, '2026-04-01', freq='MS')
master     = pd.DataFrame(index=master_idx)

print("=" * 55)
print("STEP 1: FRED pulls")
print("=" * 55)

fred_map = {
    'CPIAUCSL'        : 'cpi_us',
    'FEDFUNDS'        : 'fed_funds_rate',
    'DTWEXBGS'        : 'dxy',
    'VIXCLS'          : 'vix',
    'GS10'            : 'us_10y_yield',
    'DCOILBRENTEU'    : 'brent',
    'WALCL'           : 'fed_balance_sheet',
    'IRSTCI01INM156N' : 'india_stir',
    'INDCPIALLMINMEI' : 'cpi_india',
    'INTDSRINM193N'   : 'rbi_repo_rate',
}

for sid, col in fred_map.items():
    try:
        s = fred.get_series(sid, observation_start=START, observation_end=END)
        s = to_ms(s).reindex(master_idx)
        master[col] = s
        print(f"  ✓ {col:<28} ends {s.dropna().index[-1].strftime('%b %Y')}")
    except Exception as e:
        print(f"  ✗ {col:<28} FAILED — {e}")

# DXY backfill
try:
    dxy_n = fred.get_series('DTWEXM', observation_start=START,
                             observation_end='2006-01-01')
    dxy_n = to_ms(dxy_n).reindex(master_idx)
    master['dxy'] = master['dxy'].combine_first(dxy_n)
    print(f"  ✓ dxy backfilled to Jan 2000")
except Exception as e:
    print(f"  ⚠ dxy backfill — {e}")

# Patch repo rate gap
gap = master['rbi_repo_rate'].isnull()
master.loc[gap, 'rbi_repo_rate'] = master.loc[gap, 'india_stir']
print(f"  ✓ rbi_repo_rate patched ({gap.sum()} months from india_stir)")

print()
print("=" * 55)
print("STEP 2: RBI DBIE files")
print("=" * 55)

# ── INR/USD ───────────────────────────────────────────────────
try:
    wb = openpyxl.load_workbook(f'{RAW}rbi_inr_usd.xlsx', data_only=True)
    rows = [(r[0], float(r[4])) for r in wb.active.iter_rows(min_row=8, values_only=True)
            if isinstance(r[0], datetime) and isinstance(r[4], (int,float))]
    s = to_ms(pd.Series({r[0]:r[1] for r in rows})).reindex(master_idx)
    master['inr_usd'] = s
    print(f"  ✓ inr_usd                      ends {s.dropna().index[-1].strftime('%b %Y')}")
except Exception as e:
    print(f"  ✗ inr_usd — {e}")

# ── REER/NEER ─────────────────────────────────────────────────
try:
    wb   = openpyxl.load_workbook(f'{RAW}rbi_reer.xlsx', data_only=True)
    rows = [(r[0], float(r[1]), float(r[2])) for r in wb.active.iter_rows(min_row=7, values_only=True)
            if isinstance(r[0], datetime) and isinstance(r[2], (int,float))]
    df_r = pd.DataFrame(rows, columns=['date','neer','reer']).set_index('date')
    df_r.index = pd.to_datetime(df_r.index)
    df_r = df_r[df_r.index >= START].resample('MS').last()
    df_r.index = df_r.index.to_period('M').to_timestamp()
    master['reer'] = df_r['reer'].reindex(master_idx)
    master['neer'] = df_r['neer'].reindex(master_idx)
    print(f"  ✓ reer/neer                    ends {df_r.dropna().index[-1].strftime('%b %Y')}")
except Exception as e:
    print(f"  ✗ reer/neer — {e}")

# ── Reserves ──────────────────────────────────────────────────
try:
    wb = openpyxl.load_workbook(f'{RAW}rbi_reserves.xlsx', data_only=True)
    rows = []
    cy   = None
    for row in wb.active.iter_rows(min_row=6, values_only=True):
        yv, mv, tv = row[0], row[1], row[12]
        if isinstance(yv, str) and yv.strip().isdigit():
            cy = int(yv.strip())
        if cy and isinstance(mv, str):
            m = month_map.get(mv.strip().upper())
            if m and isinstance(tv, (int,float)):
                rows.append((pd.Timestamp(year=cy, month=m, day=1), float(tv)))
    s = to_ms(pd.Series({r[0]:r[1] for r in rows})).reindex(master_idx)
    master['fx_reserves_usd_mn'] = s
    print(f"  ✓ fx_reserves                  ends {s.dropna().index[-1].strftime('%b %Y')}")
except Exception as e:
    print(f"  ✗ fx_reserves — {e}")

# ── CAB (quarterly → monthly ffill) ───────────────────────────
try:
    wb  = openpyxl.load_workbook(f'{RAW}rbi_cab.xlsx', data_only=True)
    qmm = {'Q1':4,'Q2':7,'Q3':10,'Q4':1}
    rows = []
    cfy = None
    for row in wb.active.iter_rows(min_row=6, values_only=True):
        fy, q, tx, cab = row[0], row[1], row[2], row[4]
        if isinstance(fy, str) and '-' in fy:
            cfy = fy.strip()
        if (cfy and isinstance(q, str) and q.strip() in qmm and
                isinstance(tx, str) and tx.strip().lower() == 'net' and
                isinstance(cab, (int,float))):
            mo = qmm[q.strip()]
            yr = int(cfy.split('-')[0])
            yr = yr if mo >= 4 else yr + 1
            rows.append((pd.Timestamp(year=yr, month=mo, day=1), float(cab)))
    s = pd.Series({r[0]:r[1] for r in rows}).sort_index()
    s = s[s.index >= START].resample('MS').last().ffill()
    s.index = s.index.to_period('M').to_timestamp()
    master['current_account'] = s.reindex(master_idx)
    print(f"  ✓ current_account              ends {s.dropna().index[-1].strftime('%b %Y')}")
except Exception as e:
    print(f"  ✗ current_account — {e}")

# ── FPI flows ─────────────────────────────────────────────────
try:
    wb  = openpyxl.load_workbook(f'{RAW}rbi_fpi_flows.xlsx', data_only=True)
    cms = ['Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar']
    rows = []
    for row in wb.active.iter_rows(min_row=7, values_only=True):
        fl = row[0]
        if not isinstance(fl, str) or '-' not in fl:
            continue
        try:
            fys = int(fl.strip().split('-')[0])
        except:
            continue
        for i, v in enumerate(list(row[1:13])):
            if v is None:
                continue
            mo = month_map[cms[i].upper()]
            yr = fys if mo >= 4 else fys + 1
            rows.append((pd.Timestamp(year=yr, month=mo, day=1), float(v)))
    fpi_inr = pd.Series({r[0]:r[1] for r in rows}).sort_index()
    fpi_inr = fpi_inr[fpi_inr.index >= START].resample('MS').last()
    fpi_inr.index = fpi_inr.index.to_period('M').to_timestamp()
    master['fpi_flows_inr_cr'] = fpi_inr.reindex(master_idx)
    # Convert to USD using inr_usd
    fpi_usd = (fpi_inr * 10) / master['inr_usd']
    master['fpi_flows_usd_mn'] = fpi_usd.reindex(master_idx)
    print(f"  ✓ fpi_flows                    ends {fpi_inr.dropna().index[-1].strftime('%b %Y')}")
except Exception as e:
    print(f"  ✗ fpi_flows — {e}")

# ── Trade ─────────────────────────────────────────────────────
try:
    wb  = openpyxl.load_workbook(f'{RAW}rbi_trade.xlsx', data_only=True)
    rows = []
    cfy = None
    for row in wb.active.iter_rows(min_row=6, values_only=True):
        fv, mv, ex, im = row[0], row[1], row[2], row[3]
        if isinstance(fv, str) and '-' in fv:
            cfy = fv.strip()
        if (cfy and isinstance(mv, str) and
                isinstance(ex, (int,float)) and isinstance(im, (int,float))):
            m = month_map.get(mv.strip().upper())
            if m:
                fys = int(cfy.split('-')[0])
                yr  = fys if m >= 4 else fys + 1
                rows.append((pd.Timestamp(year=yr, month=m, day=1),
                             float(ex), float(im)))
    df_t = pd.DataFrame(rows, columns=['date','exports_usd_mn','imports_usd_mn']
                        ).set_index('date').sort_index()
    df_t = df_t[df_t.index >= START].resample('MS').last()
    df_t.index = df_t.index.to_period('M').to_timestamp()
    master['exports_usd_mn'] = df_t['exports_usd_mn'].reindex(master_idx)
    master['imports_usd_mn'] = df_t['imports_usd_mn'].reindex(master_idx)
    print(f"  ✓ trade                        ends {df_t.dropna().index[-1].strftime('%b %Y')}")
except Exception as e:
    print(f"  ✗ trade — {e}")

print()
print("=" * 55)
print("STEP 3: World Bank series")
print("=" * 55)

def load_wb_annual(fname):
    s = pd.read_csv(f'{RAW}{fname}', index_col=0, parse_dates=True).squeeze()
    s.index = pd.to_datetime(s.index).to_period('M').to_timestamp()
    s = s[~s.index.duplicated(keep='last')]
    return s.reindex(master_idx).ffill()

master['gdp_india_usd']   = load_wb_annual('wb_gdp_india.csv')
master['gdp_us_usd']      = load_wb_annual('wb_gdp_us.csv')
master['gdp_pc_india']    = load_wb_annual('wb_gdp_pc_india.csv')
master['gdp_pc_us']       = load_wb_annual('wb_gdp_pc_us.csv')
master['remittances_usd'] = load_wb_annual('wb_remittances.csv')
print(f"  ✓ World Bank annual series loaded and forward-filled")

print()
print("=" * 55)
print("STEP 4: CPI India base year splice")
print("=" * 55)

# MOSPI data (base 2024=100) — provided by user
mospi_new = {
    '2025-01-01': 101.67, '2025-02-01': 101.32, '2025-03-01': 101.39,
    '2025-04-01': 101.58, '2025-05-01': 101.90, '2025-06-01': 102.51,
    '2025-07-01': 103.35, '2025-08-01': 103.74, '2025-09-01': 103.74,
    '2025-10-01': 103.74, '2025-11-01': 104.01, '2025-12-01': 104.10,
    '2026-01-01': 104.45, '2026-02-01': 104.57, '2026-03-01': 104.84,
    '2026-04-01': 105.12,
}
mospi_yoy = {
    '2026-01-01': 2.74, '2026-02-01': 3.21,
    '2026-03-01': 3.40, '2026-04-01': 3.48,
}

# Find overlap months (have FRED CPI AND in mospi_new)
cpi_end  = master['cpi_india'].dropna().index[-1]
print(f"  FRED CPI ends: {cpi_end.strftime('%b %Y')}")

overlap  = [(d, get_val(master, pd.Timestamp(d), 'cpi_india'))
            for d in sorted(mospi_new.keys())
            if pd.Timestamp(d) <= cpi_end]
overlap  = [(d, v) for d, v in overlap if not np.isnan(v)]

if len(overlap) == 0:
    # FRED ends before MOSPI starts — use last known FRED value
    # and apply YoY from MOSPI for 2025 months
    print(f"  No direct overlap — using last FRED value to anchor")
    # Estimate 2024 MOSPI values by working backward from 2025 using
    # approximate YoY of 4.5% (RBI reported avg for 2024)
    last_fred_val  = master['cpi_india'].dropna().iloc[-1]
    last_fred_date = master['cpi_india'].dropna().index[-1]
    print(f"  Last FRED value: {last_fred_date.strftime('%b %Y')} = {last_fred_val:.3f}")
    # Anchor ratio using the last available month
    # Find corresponding MOSPI month
    last_str = last_fred_date.strftime('%Y-%m-01')
    if last_str in mospi_new:
        ratio = last_fred_val / mospi_new[last_str]
    else:
        # Interpolate: find nearest MOSPI date
        mospi_dates = sorted(mospi_new.keys())
        nearest = min(mospi_dates,
                      key=lambda d: abs(pd.Timestamp(d) - last_fred_date))
        ratio = last_fred_val / mospi_new[nearest]
    print(f"  Splice ratio: {ratio:.4f}")
else:
    ratios = [v / mospi_new[d] for d, v in overlap]
    ratio  = np.mean(ratios)
    print(f"  Overlap months: {len(overlap)}")
    for d, v in overlap:
        print(f"    {d[:7]}  FRED={v:.3f}  MOSPI={mospi_new[d]:.2f}  ratio={v/mospi_new[d]:.4f}")
    print(f"  Splice ratio: {ratio:.4f}  (std={np.std(ratios):.5f})")

# Fill CPI for all MOSPI months not yet in master
filled = 0
for date_str in sorted(mospi_new.keys()):
    dt = pd.Timestamp(date_str)
    if dt not in master.index:
        continue
    if pd.notna(get_val(master, dt, 'cpi_india')):
        continue  # already have it

    if date_str in mospi_yoy:
        dt_12 = dt - pd.DateOffset(months=12)
        prev  = get_val(master, dt_12, 'cpi_india')
        if not np.isnan(prev):
            master.loc[dt, 'cpi_india'] = prev * (1 + mospi_yoy[date_str] / 100)
            filled += 1
            continue

    master.loc[dt, 'cpi_india'] = mospi_new[date_str] * ratio
    filled += 1

print(f"  Filled {filled} months via splice")
print(f"  CPI India now ends: {master['cpi_india'].dropna().index[-1].strftime('%b %Y')}")
print(f"  Last 6 values:")
print(master['cpi_india'].dropna().tail(6).round(3).to_string())

print()
print("=" * 55)
print("STEP 5: Derived series")
print("=" * 55)

master['cpi_india_yoy']          = master['cpi_india'].pct_change(12, fill_method=None) * 100
master['cpi_us_yoy']             = master['cpi_us'].pct_change(12, fill_method=None) * 100
master['inflation_diff']         = master['cpi_india_yoy'] - master['cpi_us_yoy']
master['real_rate_india']        = master['rbi_repo_rate'] - master['cpi_india_yoy']
master['real_rate_us']           = master['fed_funds_rate'] - master['cpi_us_yoy']
master['real_rate_diff']         = master['real_rate_india'] - master['real_rate_us']
master['prod_india_yoy']         = master['gdp_pc_india'].pct_change(12, fill_method=None) * 100
master['prod_us_yoy']            = master['gdp_pc_us'].pct_change(12, fill_method=None) * 100
master['prod_diff']              = master['prod_india_yoy'] - master['prod_us_yoy']
master['justified_depreciation'] = master['inflation_diff'] - master['prod_diff']
master['trade_balance']          = master['exports_usd_mn'] - master['imports_usd_mn']
master['import_cover']           = master['fx_reserves_usd_mn'] / master['imports_usd_mn']
master['brent_yoy']              = master['brent'].pct_change(12, fill_method=None) * 100
master['cab_pct_gdp']            = ((master['current_account'] * 4) /
                                    (master['gdp_india_usd'] / 1e6)) * 100
master['remit_pct_gdp']          = (master['remittances_usd'] /
                                    master['gdp_india_usd']) * 100
master['fed_bs_yoy']             = master['fed_balance_sheet'].pct_change(12, fill_method=None) * 100
master['inr_yoy_chg']            = master['inr_usd'].pct_change(12, fill_method=None) * 100
master['reserves_mom_chg']       = master['fx_reserves_usd_mn'].diff()

print("  ✓ All 19 derived series computed")

print()
print("=" * 55)
print("STEP 6: Validation")
print("=" * 55)
print(f"  Shape      : {master.shape}")
print(f"  Date range : {master.index[0].strftime('%b %Y')} → "
      f"{master.index[-1].strftime('%b %Y')}")
print()

nulls = master.isnull().sum()
for col, n in nulls.items():
    pct  = n / len(master) * 100
    flag = '⚠ ' if pct > 20 else '  '
    end  = master[col].dropna().index[-1].strftime('%b %Y') \
           if master[col].dropna().shape[0] > 0 else 'N/A'
    print(f"  {flag}{col:<30} {n:>3} nulls ({pct:.0f}%)  ends {end}")

print()
print("  CPI India YoY — last 8 months (expect 3–6%):")
print(master['cpi_india_yoy'].dropna().tail(8).round(2).to_string())
print()
print("  Spot checks:")
print(f"    INR/USD Apr 2026 : {master.loc['2026-04-01','inr_usd']:.2f}  (expect ~84-86)")
print(f"    Brent Jan 2026   : {master.loc['2026-01-01','brent']:.2f}  (expect ~75-80)")
print(f"    Fed funds Jan 26 : {master.loc['2026-01-01','fed_funds_rate']:.2f}  (expect ~4.3-4.5)")
print(f"    Import cover     : {master.loc['2026-01-01','import_cover']:.1f} months")

master.to_csv(f'{PROCESSED}master.csv')
print(f"\n✓ master.csv saved → {PROCESSED}master.csv")
print(f"  {master.shape[0]} months × {master.shape[1]} columns")

STEP 1: FRED pulls
  ✓ cpi_us                       ends Apr 2026
  ✓ fed_funds_rate               ends Apr 2026
  ✓ dxy                          ends Apr 2026
  ✓ vix                          ends Apr 2026
  ✓ us_10y_yield                 ends Apr 2026
  ✓ brent                        ends Apr 2026
  ✓ fed_balance_sheet            ends Apr 2026
  ✓ india_stir                   ends Mar 2026
  ✓ cpi_india                    ends Mar 2025
  ✓ rbi_repo_rate                ends Jul 2022
  ✓ dxy backfilled to Jan 2000
  ✓ rbi_repo_rate patched (45 months from india_stir)

STEP 2: RBI DBIE files
  ✓ inr_usd                      ends Apr 2026
  ✓ reer/neer                    ends Apr 2026
  ✓ fx_reserves                  ends Feb 2026
  ✓ current_account              ends Oct 2025
  ✓ fpi_flows                    ends Jun 2025
  ✓ trade                        ends Mar 2026

STEP 3: World Bank series
  ✓ World Bank annual series loaded and forward-filled

STEP 4: CPI India base year splice
  

In [1]:
# ============================================================
# Phase 1 | Cell 7: Missing Value Treatment + Model Window
# ============================================================

import pandas as pd
import numpy as np

PROCESSED = '../data/processed/'

master = pd.read_csv(f'{PROCESSED}master.csv', index_col=0, parse_dates=True)
master.index = pd.to_datetime(master.index).to_period('M').to_timestamp()
master = master[~master.index.duplicated(keep='last')]

print("=" * 55)
print("STEP 1: Targeted imputations on master.csv")
print("=" * 55)

# ── Fix 1: CPI US — BLS shutdown Oct 2025 ────────────────────
# Linear interpolation between Sep 2025 and Nov 2025
# This is the only statistically defensible approach for a
# known institutional gap — do NOT ffill (that assumes no change)
before = master.loc['2025-10-01', 'cpi_us']
master['cpi_us'] = master['cpi_us'].interpolate(method='linear')
after  = master.loc['2025-10-01', 'cpi_us']
print(f"  ✓ CPI US Oct 2025: NaN → {after:.3f} (interpolated)")
print(f"    Sep 2025: {master.loc['2025-09-01','cpi_us']:.3f}")
print(f"    Oct 2025: {after:.3f}  ← interpolated")
print(f"    Nov 2025: {master.loc['2025-11-01','cpi_us']:.3f}")

# ── Fix 2: CPI India — check for any remaining NaNs ──────────
cpi_nulls = master['cpi_india']['2004-04-01':].isnull().sum()
if cpi_nulls > 0:
    print(f"\n  ⚠ CPI India has {cpi_nulls} nulls in post-2004 window")
    print(f"    Null dates: {list(master['cpi_india']['2004-04-01':][master['cpi_india']['2004-04-01':].isnull()].index.strftime('%b %Y'))}")
    # These are the Aug-Oct 2025 flat MOSPI months
    # Use linear interpolation — same logic as CPI US
    master['cpi_india'] = master['cpi_india'].interpolate(method='linear')
    print(f"    ✓ Patched via linear interpolation")
else:
    print(f"\n  ✓ CPI India: no nulls in post-2004 window")

# ── Fix 3: Recompute CPI-dependent series after patches ──────
master['cpi_india_yoy']   = master['cpi_india'].pct_change(12, fill_method=None) * 100
master['cpi_us_yoy']      = master['cpi_us'].pct_change(12, fill_method=None) * 100
master['inflation_diff']  = master['cpi_india_yoy'] - master['cpi_us_yoy']
master['real_rate_india'] = master['rbi_repo_rate'] - master['cpi_india_yoy']
master['real_rate_us']    = master['fed_funds_rate'] - master['cpi_us_yoy']
master['real_rate_diff']  = master['real_rate_india'] - master['real_rate_us']
master['justified_depreciation'] = master['inflation_diff'] - master['prod_diff']
print(f"\n  ✓ CPI-dependent series recomputed")

# ── Fix 4: Current account — verify ffill is clean ───────────
# Any nulls after Apr 2000 (first CAB data point) should not exist
# if ffill worked correctly in Cell 6
cab_nulls = master['current_account']['2000-04-01':'2025-10-01'].isnull().sum()
if cab_nulls > 0:
    print(f"\n  ⚠ Current account: {cab_nulls} nulls found — applying ffill")
    master['current_account'] = master['current_account'].ffill()
    master['cab_pct_gdp'] = ((master['current_account'] * 4) /
                             (master['gdp_india_usd'] / 1e6)) * 100
else:
    print(f"  ✓ Current account: ffill clean")

# ── Save patched master.csv (full reference) ──────────────────
master.to_csv(f'{PROCESSED}master.csv')
print(f"\n  ✓ master.csv overwritten (full reference, Jan 2000 → Apr 2026)")

print()
print("=" * 55)
print("STEP 2: Build master_model.csv (clean training window)")
print("=" * 55)

# ── Define training window ────────────────────────────────────
# Left:  Apr 2004 — REER starts (binding structural constraint)
# Right: Jun 2025 — FPI flows end (most lagged variable)
LEFT  = '2004-04-01'
RIGHT = '2025-06-01'

model = master[LEFT:RIGHT].copy()
print(f"  Window: {LEFT[:7]} → {RIGHT[:7]}  ({len(model)} months)")

# ── Null audit within training window ────────────────────────
print(f"\n  Null audit within training window:")
print(f"  {'─'*50}")

# Columns to check — exclude YoY first-12-month warmup
# (Apr 2004 → Mar 2005 will have YoY nulls — this is correct)
yoy_cols = [c for c in model.columns if 'yoy' in c or 'diff' in c
            or 'depreciation' in c]
level_cols = [c for c in model.columns if c not in yoy_cols]

level_nulls = model[level_cols].isnull().sum()
yoy_nulls   = model[yoy_cols].isnull().sum()

print(f"\n  LEVEL columns (must be 0 nulls):")
any_level_issue = False
for col, n in level_nulls.items():
    flag = '  ✓' if n == 0 else '  ✗'
    if n > 0:
        any_level_issue = True
        end = model[col].dropna().index[-1].strftime('%b %Y') if model[col].dropna().shape[0] > 0 else 'N/A'
        print(f"  {flag} {col:<30} {n:>3} nulls  (ends {end})")
    else:
        print(f"  {flag} {col:<30}   0 nulls")

print(f"\n  YoY/DIFF columns (first 12 months expected null — LEAVE AS IS):")
for col, n in yoy_nulls.items():
    warmup = model[col].isnull().sum()
    # Check if all nulls are in warmup period
    null_dates = model[col][model[col].isnull()].index
    all_warmup = all(d < pd.Timestamp('2005-04-01') for d in null_dates)
    flag = '  ✓' if all_warmup else '  ⚠'
    print(f"  {flag} {col:<30} {n:>3} nulls  {'(warmup only)' if all_warmup else '(SCATTERED — investigate)'}")

# ── Final clean check ─────────────────────────────────────────
total_nulls = model.isnull().sum().sum()
level_null_total = level_nulls.sum()

print(f"\n{'─'*55}")
print(f"  Total nulls in window    : {total_nulls}")
print(f"  Nulls in level columns   : {level_null_total}  ← must be 0")
print(f"  Nulls in YoY cols        : {yoy_nulls.sum()}  ← warmup only, acceptable")
print(f"  Window length            : {len(model)} months")

if level_null_total == 0:
    print(f"\n  ✓ GOLDEN RULE MET: Level columns are clean")
    print(f"    Safe to pass to 02_structural_anchor.ipynb")
else:
    print(f"\n  ✗ Level column nulls remain — investigate before proceeding")

# ── Save master_model.csv ─────────────────────────────────────
model.to_csv(f'{PROCESSED}master_model.csv')
print(f"\n  ✓ master_model.csv saved → {PROCESSED}master_model.csv")
print(f"    {model.shape[0]} months × {model.shape[1]} columns")
print(f"    This is your Phase 2–5 modeling dataset")

print()
print("=" * 55)
print("STEP 3: Final summary")
print("=" * 55)
print(f"""
  master.csv        Full reference   Jan 2000 → Apr 2026  316 × 42
  master_model.csv  Clean training   Apr 2004 → Jun 2025  {len(model)} × {model.shape[1]}

  What was imputed (documented):
    CPI US  Oct 2025    : linear interpolation (BLS shutdown)
    CPI India (if gap)  : linear interpolation (MOSPI lag)
    Current account     : forward-fill (quarterly → monthly)

  What was NOT touched (mathematically correct):
    YoY series first 12 months  : NaN (no prior year to reference)
    CAB step function            : flat 3-month blocks (quarterly data)
    REER pre-Apr 2004            : NaN (series didn't exist)
""")

STEP 1: Targeted imputations on master.csv
  ✓ CPI US Oct 2025: NaN → 324.654 (interpolated)
    Sep 2025: 324.245
    Oct 2025: 324.654  ← interpolated
    Nov 2025: 325.063

  ✓ CPI India: no nulls in post-2004 window

  ✓ CPI-dependent series recomputed
  ✓ Current account: ffill clean

  ✓ master.csv overwritten (full reference, Jan 2000 → Apr 2026)

STEP 2: Build master_model.csv (clean training window)
  Window: 2004-04 → 2025-06  (255 months)

  Null audit within training window:
  ──────────────────────────────────────────────────

  LEVEL columns (must be 0 nulls):
    ✓ cpi_us                           0 nulls
    ✓ fed_funds_rate                   0 nulls
    ✓ dxy                              0 nulls
    ✓ vix                              0 nulls
    ✓ us_10y_yield                     0 nulls
    ✓ brent                            0 nulls
    ✓ fed_balance_sheet                0 nulls
    ✓ india_stir                       0 nulls
    ✓ cpi_india                        0 nu